# CLM-v0.1-8B — Zero-Shot Pair Scoring on Fuzzy Candidates (Free-tier T4)

`Contrastive-LM/CLM-v0.1-8B` is a **questions head over frozen Qwen3-8B
embeddings**, so this notebook runs three processes on the single T4:

1. `vllm serve Qwen/Qwen3-8B-AWQ` — 4-bit AWQ embedding server on port 8090
   (fits in ~6 GB VRAM, leaving headroom for KV cache on 16 GB T4).
2. `clm-serve` — the CLM questions head on port 8700.
3. `eval_clm.py` — scores each fuzzy-surviving pair via
   `CLMClient.system_one(state, questions={'match': Score(...)})`.

**Data assumed on Drive at** `MyDrive/amazon-entity-resolution/`:
- `train_labeled_pairs.tsv` (already uploaded).

**Free-tier budget:** default `--limit 5000` pairs (~30–60 min end-to-end,
assuming ~2–3 pair/s). Increase if the session has time left.

**If AWQ vLLM OOMs on T4:** drop `--max-model-len` to 512 in section 3,
or switch to Colab Pro (L4/A100).

## 1. GPU + Drive + deps

In [ ]:
!nvidia-smi || echo 'NO GPU — Runtime → Change runtime type → T4 GPU'
from google.colab import drive
drive.mount('/content/drive')

# vLLM pulls a lot; pin versions that are known to work on Colab T4.
!pip install -q --upgrade pip
!pip install -q "vllm>=0.6.3" "contrastive-lm" \
    "transformers>=4.44.0" "accelerate>=0.33.0" \
    autoawq sentencepiece protobuf requests

DRIVE = '/content/drive/MyDrive/amazon-entity-resolution'
import os
assert os.path.exists(f'{DRIVE}/train_labeled_pairs.tsv'), \
    'upload train_labeled_pairs.tsv to ' + DRIVE + ' first'
!ls -lh {DRIVE}


## 2. Derive `val_pairs.tsv` from `train_labeled_pairs.tsv`

~3–5 min. Cached to Drive so re-runs skip this.

In [ ]:
%%writefile /content/sample_labeled_pairs.py
"""Build stratified 100k / 500k train subsets + a shared val split.

Sampling is by source1 entity (not by row), so all pairs belonging to one
source1 land in the same split — no leakage across train/val. The 1:3
positive:negative ratio from the full file is preserved automatically because
we take all rows for every sampled source1.

Val is 10% of source1 ids, held out before either train subset is sampled and
identical for both, so the 100k and 500k runs are directly comparable.

Usage:
    python sample_labeled_pairs.py \\
        --input  dataset/train/train_labeled_pairs.tsv \\
        --outdir dataset/train
"""
import argparse
import csv
import random
import sys
from pathlib import Path

csv.field_size_limit(sys.maxsize)

SEED = 42
VAL_FRAC = 0.10
TRAIN_SIZES = {"100k": 100_000, "500k": 500_000}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--outdir", required=True, type=Path)
    args = ap.parse_args()

    print(f"Pass 1: collecting source1 ids and row counts from {args.input.name} ...", flush=True)
    src1_rows: dict[str, int] = {}
    with args.input.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            s1 = row["source1_entity_id"]
            src1_rows[s1] = src1_rows.get(s1, 0) + 1
    print(f"  {len(src1_rows):,} distinct source1 entities", flush=True)

    rng = random.Random(SEED)
    all_ids = list(src1_rows.keys())
    rng.shuffle(all_ids)
    n_val = int(len(all_ids) * VAL_FRAC)
    val_ids = set(all_ids[:n_val])
    train_pool = all_ids[n_val:]
    print(f"  val: {len(val_ids):,} source1 ids", flush=True)

    picks: dict[str, set[str]] = {}
    for tag, target_rows in TRAIN_SIZES.items():
        chosen: set[str] = set()
        total = 0
        for s1 in train_pool:
            if total >= target_rows:
                break
            chosen.add(s1)
            total += src1_rows[s1]
        picks[tag] = chosen
        print(f"  train_{tag}: {len(chosen):,} source1 ids -> {total:,} rows", flush=True)

    outs = {
        "val": args.outdir / "val_pairs.tsv",
        "100k": args.outdir / "train_pairs_100k.tsv",
        "500k": args.outdir / "train_pairs_500k.tsv",
    }
    print("Pass 2: writing splits ...", flush=True)
    with args.input.open("r", encoding="utf-8", newline="") as f, \
            outs["val"].open("w", encoding="utf-8", newline="") as fv, \
            outs["100k"].open("w", encoding="utf-8", newline="") as f100, \
            outs["500k"].open("w", encoding="utf-8", newline="") as f500:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)
        for w in (fv, f100, f500):
            csv.writer(w, delimiter="\t", lineterminator="\n").writerow(header)
        wv = csv.writer(fv, delimiter="\t", lineterminator="\n")
        w100 = csv.writer(f100, delimiter="\t", lineterminator="\n")
        w500 = csv.writer(f500, delimiter="\t", lineterminator="\n")
        counts = {"val": 0, "100k": 0, "500k": 0}
        for row in reader:
            s1 = row[0]
            if s1 in val_ids:
                wv.writerow(row); counts["val"] += 1
            if s1 in picks["100k"]:
                w100.writerow(row); counts["100k"] += 1
            if s1 in picks["500k"]:
                w500.writerow(row); counts["500k"] += 1
    for tag, n in counts.items():
        print(f"  {outs[tag].name}: {n:,} rows", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import os
if not os.path.exists(f'{DRIVE}/val_pairs.tsv'):
    !python /content/sample_labeled_pairs.py \
        --input  {DRIVE}/train_labeled_pairs.tsv \
        --outdir {DRIVE}
!ls -lh {DRIVE}/val_pairs.tsv && wc -l {DRIVE}/val_pairs.tsv

## 3. Launch the Qwen3-8B-AWQ embedding server (vLLM, background)

Business records are short so `--max-model-len 1024` is plenty and
keeps the KV cache tiny. Logs stream to `/content/vllm.log`.

In [ ]:
import subprocess, time, os, signal, requests

VLLM_LOG = '/content/vllm.log'
vllm_cmd = [
    'vllm', 'serve', 'Qwen/Qwen3-8B-AWQ',
    '--served-model-name', 'qwen3-8b',
    '--runner', 'pooling',
    '--quantization', 'awq',
    '--dtype', 'float16',
    '--max-model-len', '1024',
    '--gpu-memory-utilization', '0.80',
    '--port', '8090',
]
print('launching:', ' '.join(vllm_cmd))
vllm_proc = subprocess.Popen(vllm_cmd,
    stdout=open(VLLM_LOG, 'w'), stderr=subprocess.STDOUT,
    preexec_fn=os.setsid)
print('vllm pid:', vllm_proc.pid, '— tailing log until /v1/models responds')

for i in range(180):  # up to ~15 min for first-time model download
    time.sleep(5)
    try:
        r = requests.get('http://127.0.0.1:8090/v1/models', timeout=2)
        if r.ok:
            print('vllm ready:', r.json()); break
    except Exception:
        pass
    if i % 6 == 0:
        !tail -n 5 {VLLM_LOG}
else:
    raise RuntimeError('vllm did not come up in time; see /content/vllm.log')


## 4. Launch `clm-serve` (background) and probe end-to-end

Points at the local vLLM. The probe issues one `system_one` call so we
fail fast if anything is misconfigured before the long eval loop.

In [ ]:
CLM_LOG = '/content/clm.log'
clm_env = {**os.environ,
    'CLM_EMB_URL': 'http://127.0.0.1:8090/v1/embeddings',
    'EMB_URL':     'http://127.0.0.1:8090/v1/embeddings',
}
clm_proc = subprocess.Popen(['clm-serve'],
    stdout=open(CLM_LOG, 'w'), stderr=subprocess.STDOUT,
    env=clm_env, preexec_fn=os.setsid)
print('clm-serve pid:', clm_proc.pid)

for i in range(60):
    time.sleep(2)
    try:
        r = requests.get('http://127.0.0.1:8700/', timeout=2)
        if r.status_code < 500:
            print('clm-serve ready:', r.status_code); break
    except Exception:
        pass
else:
    !tail -n 40 {CLM_LOG}
    raise RuntimeError('clm-serve did not come up; see /content/clm.log')

# End-to-end probe.
from clm import CLMClient, Score
try:
    client = CLMClient(base_url='http://127.0.0.1:8700')
except TypeError:
    client = CLMClient()
probe = client.system_one(
    state='Record A: Acme Corp | 123 Main St, NYC\n'
          'Record B: Acme Corporation | 123 Main Street, New York',
    questions={'match': Score(
        instructions='Are these the same business entity?',
        criteria=['Definitely different', 'Possibly same', 'Definitely same'],
    )},
)
print('probe probabilities:', probe.answers['match'].probabilities)


## 5. Write the eval script

In [ ]:
%%writefile /content/eval_clm.py
"""Benchmark Contrastive-LM/CLM-v0.1-8B on the fuzzy candidate pairs, no fine-tuning.

CLM is a "questions head" over frozen Qwen3-8B embeddings. It expects a running
Qwen3 embedding server (vLLM) and a running `clm-serve` process — the notebook
in `make_clm_colab.py` starts both.

Per pair we ask a `Score` question ("are these the same business?") via
`CLMClient.system_one` and combine the returned criterion probabilities into a
scalar in [0, 1]. Same downstream metrics as eval_cross_encoder.py so results
are directly comparable.

Usage (once vllm + clm-serve are up):
    python eval_clm.py \\
        --val   /content/drive/MyDrive/amazon-entity-resolution/val_pairs.tsv \\
        --out   /content/drive/MyDrive/amazon-entity-resolution/eval_clm_8b.json \\
        --limit 5000
"""
import argparse
import csv
import json
import os
import sys
import time
from collections import defaultdict
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

csv.field_size_limit(sys.maxsize)

CRITERIA = ["Definitely different", "Possibly same", "Definitely same"]
# Weighted collapse of the 3 criterion probabilities into a single [0, 1] score.
CRITERION_WEIGHTS = [0.0, 0.5, 1.0]
INSTRUCTIONS = (
    "Decide whether the two business records below refer to the same physical "
    "business entity. Consider name variants, abbreviations, translations, and "
    "address formatting differences."
)


def side_text(name: str, addr: str) -> str:
    return f"{name} | {addr or 'no address'}"


def pair_state(row: dict) -> str:
    a = side_text(row["source1_business_name"], row["source1_business_address"])
    b = side_text(row["candidate_business_name"], row["candidate_business_address"])
    return f"Record A: {a}\nRecord B: {b}"


def sweep_thresholds(scores, labels):
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    tp = fp = 0
    P = sum(labels)
    best = (0.5, 0.0, 0.0, 0.0)
    for idx in order:
        if labels[idx] == 1:
            tp += 1
        else:
            fp += 1
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / P if P else 0.0
        if precision + recall == 0:
            continue
        f05 = 1.25 * precision * recall / (0.25 * precision + recall)
        if f05 > best[3]:
            best = (float(scores[idx]), precision, recall, f05)
    return {"threshold": best[0], "precision": best[1], "recall": best[2], "f0_5": best[3]}


def fixed_threshold(scores, labels, thr):
    tp = fp = fn = 0
    for s, y in zip(scores, labels):
        pred = 1 if s >= thr else 0
        if pred == 1 and y == 1: tp += 1
        elif pred == 1 and y == 0: fp += 1
        elif pred == 0 and y == 1: fn += 1
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    if precision + recall == 0:
        return {"threshold": thr, "precision": 0.0, "recall": 0.0, "f0_5": 0.0}
    f05 = 1.25 * precision * recall / (0.25 * precision + recall)
    return {"threshold": thr, "precision": precision, "recall": recall, "f0_5": f05}


def source1_macro_f05(pair_records, scores, thr):
    truth_by_s1: dict[str, set[str]] = defaultdict(set)
    preds_by_s1: dict[str, set[str]] = defaultdict(set)
    all_s1: set[str] = set()
    for rec, s in zip(pair_records, scores):
        s1 = rec["source1_entity_id"]
        cid = rec["candidate_entity_id"]
        all_s1.add(s1)
        if int(rec["label"]) == 1:
            truth_by_s1[s1].add(cid)
        if s >= thr:
            preds_by_s1[s1].add(cid)
    f05s = []
    for s1 in all_s1:
        pred = preds_by_s1.get(s1, set())
        true = truth_by_s1.get(s1, set())
        if not true and not pred:
            f05s.append(1.0); continue
        if not pred:
            f05s.append(0.0); continue
        tp = len(pred & true)
        precision = tp / len(pred) if pred else 0.0
        recall = tp / len(true) if true else 0.0
        if precision + recall == 0:
            f05s.append(0.0); continue
        f05s.append(1.25 * precision * recall / (0.25 * precision + recall))
    return sum(f05s) / len(f05s) if f05s else 0.0


def collapse(probs) -> float:
    """Weighted mean of criterion probabilities → [0, 1]."""
    return float(sum(p * w for p, w in zip(probs, CRITERION_WEIGHTS)))


def score_pair(client, question_ctor, row: dict) -> float:
    """One system_one call. Returns 0.5 (neutral) on transient failure."""
    q = question_ctor(instructions=INSTRUCTIONS, criteria=CRITERIA)
    for attempt in range(2):
        try:
            r = client.system_one(state=pair_state(row), questions={"match": q})
            return collapse(r.answers["match"].probabilities)
        except Exception as e:
            if attempt == 0:
                time.sleep(1.0); continue
            print(f"  warn: pair failed twice, defaulting to 0.5 ({e})", flush=True)
            return 0.5
    return 0.5


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--val", required=True, type=Path)
    ap.add_argument("--out", required=True, type=Path)
    ap.add_argument("--clm-url", default="http://127.0.0.1:8700",
                    help="Where clm-serve is listening.")
    ap.add_argument("--limit", type=int, default=None,
                    help="Cap pairs (recommended for free-tier Colab: 5000).")
    ap.add_argument("--log-every", type=int, default=50)
    args = ap.parse_args()

    from clm import CLMClient, Score  # noqa: E402

    # CLMClient may accept a base URL as positional/kwarg depending on version;
    # try the common forms and fall back to default localhost binding.
    client = None
    for kw in ({"base_url": args.clm_url}, {"url": args.clm_url}, {}):
        try:
            client = CLMClient(**kw); break
        except TypeError:
            continue
    if client is None:
        client = CLMClient()

    print(f"loading val: {args.val}", flush=True)
    pair_records: list[dict] = []
    with args.val.open("r", encoding="utf-8", newline="") as f:
        for i, r in enumerate(csv.DictReader(f, delimiter="\t")):
            if args.limit is not None and i >= args.limit:
                break
            pair_records.append(r)
    print(f"  val pairs: {len(pair_records):,}", flush=True)

    scores: list[float] = []
    t0 = time.time()
    for i, rec in enumerate(pair_records):
        scores.append(score_pair(client, Score, rec))
        if (i + 1) % args.log_every == 0:
            speed = (i + 1) / max(time.time() - t0, 1e-6)
            eta = (len(pair_records) - i - 1) / max(speed, 1e-6)
            print(f"  scored {i+1:,}/{len(pair_records):,} "
                  f"({speed:.2f} pair/s, ETA {eta/60:.1f} min)", flush=True)
    dur = time.time() - t0
    print(f"scoring done in {dur/60:.1f} min ({len(pair_records)/dur:.2f} pair/s).", flush=True)

    labels = [int(r["label"]) for r in pair_records]
    metrics = {
        "pair_at_0.5": fixed_threshold(scores, labels, 0.5),
        "pair_best_threshold": sweep_thresholds(scores, labels),
    }
    for thr in (0.3, 0.5, 0.7, metrics["pair_best_threshold"]["threshold"]):
        metrics[f"source1_macro_f0_5_at_{thr:.3f}"] = source1_macro_f05(pair_records, scores, thr)

    result = {
        "model": "Contrastive-LM/CLM-v0.1-8B",
        "api": "system_one/Score",
        "criteria": CRITERIA,
        "weights": CRITERION_WEIGHTS,
        "val": str(args.val),
        "n_pairs": len(pair_records),
        "n_pos": sum(labels),
        "n_neg": len(labels) - sum(labels),
        "n_source1_in_val": len({r["source1_entity_id"] for r in pair_records}),
        "seconds_total": dur,
        "pairs_per_second": len(pair_records) / dur if dur else 0.0,
        "metrics": metrics,
    }
    args.out.write_text(json.dumps(result, indent=2))
    print(f"wrote {args.out}", flush=True)
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


## 6. Benchmark on 5 000 pairs (~30–60 min)

One `system_one` call per pair. If pair/s is healthy after a few hundred,
cancel and re-run with a larger `--limit` (or drop it entirely).

In [ ]:
!python /content/eval_clm.py \
    --val   {DRIVE}/val_pairs.tsv \
    --out   {DRIVE}/eval_clm_8b.json \
    --clm-url http://127.0.0.1:8700 \
    --limit 5000
!cat {DRIVE}/eval_clm_8b.json

## 7. Compare against previous baselines on Drive

In [ ]:
import json, glob
for p in sorted(glob.glob(f'{DRIVE}/eval_*.json')):
    d = json.load(open(p))
    m = d.get('metrics', {})
    best = m.get('pair_best_threshold', {})
    print(f"{p.split('/')[-1]:<32} "
          f"pair@0.5 F0.5={m.get('pair_at_0.5', {}).get('f0_5', 0):.3f}  "
          f"best F0.5={best.get('f0_5', 0):.3f} @ thr={best.get('threshold', 0):.3f}  "
          f"s1-macro@0.5={m.get('source1_macro_f0_5_at_0.500', 0):.3f}")


## 8. Cleanup (optional)

Kill the background servers before disconnecting so you don't leave GPU
memory pinned mid-session.

In [ ]:
import os, signal
for p in (clm_proc, vllm_proc):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except Exception as e:
        print('kill failed:', e)
print('done')
